In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1rvCShRNyfEqrSOZwwBPygvfyFmJ-6Z-5", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!

import torch
import torch.nn as nn
import sys
import numpy as np
import matplotlib.pyplot as plt

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU available: {gpu_name}")
    print(f"   Memory: {gpu_mem:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. Some cells require a GPU.")
    print("   Go to Runtime -> Change runtime type -> GPU")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why This Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_this_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# Activation Memory and Mixed Precision Training -- Vizuara

## 1. Why Does This Matter?

In Notebook 1, we learned that weights, gradients, and optimizer states consume 16 bytes per parameter. For a 7B model, that is about 104 GB -- already more than a single A100.

But we left out the fourth component: **activations**. These are the intermediate outputs saved at each layer during the forward pass so they can be reused during backpropagation. Unlike the other components, activation memory scales with **batch size and sequence length**, not just parameter count.

In this notebook, we will:
- Measure activation memory empirically
- Verify the theoretical formula from the Megatron-LM paper
- Implement mixed precision training and measure its memory savings

**What you will build:** An empirical activation profiler and a working mixed precision training loop.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition

Imagine you are reading a long textbook and taking notes. At each chapter, you write down the key results so you can reference them later when solving the final problem set.

Activations are exactly those notes. During the forward pass, each layer produces an output tensor. The backward pass needs these outputs to compute gradients (by the chain rule). So PyTorch saves them in memory.

The crucial insight: if you double the batch size, you double the activation memory. If you double the sequence length, you (roughly) double it again. This makes activations the **most tunable** memory component -- just reduce batch size.

In [ ]:
#@title 🎧 Listen: The Mathematics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_the_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics

For a transformer model, the activation memory per layer includes:
- Layer norm outputs: $B \times S \times H$ (2 per layer)
- Attention Q, K, V projections: $3 \times B \times S \times H$
- Attention scores: $B \times A \times S \times S$
- Attention output: $B \times S \times H$
- FFN intermediate: $B \times S \times 4H$
- FFN output: $B \times S \times H$

The Megatron-LM approximation sums these up:

$$M_{\text{activations}} \approx L \times B \times S \times H \times \left(34 + 5 \cdot \frac{A \cdot S}{H}\right) \text{ bytes}$$

When $\frac{A \cdot S}{H}$ is small (typical for large hidden dims), this simplifies to:

$$M_{\text{activations}} \approx L \times B \times S \times H \times 34 \text{ bytes}$$

In [ ]:
#@title 🎧 Code Walkthrough: Theoretical Calculator
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_theoretical_calculator.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Theoretical activation memory calculator

def activation_memory_gb(num_layers, batch_size, seq_len, hidden_dim,
                          num_heads, include_attn_scores=True):
    """Estimate activation memory using the Megatron-LM formula.

    Args:
        num_layers: Number of transformer layers
        batch_size: Training batch size
        seq_len: Sequence length
        hidden_dim: Hidden dimension
        num_heads: Number of attention heads
        include_attn_scores: Include the B*A*S*S attention score term
    """
    if include_attn_scores:
        bytes_per_elem = 34 + 5 * (num_heads * seq_len / hidden_dim)
    else:
        bytes_per_elem = 34

    total_bytes = num_layers * batch_size * seq_len * hidden_dim * bytes_per_elem
    return total_bytes / (1024**3)

# Calculate for common configurations
configs = [
    ("GPT-2 (124M)", 12, 768, 12, 1024),
    ("LLaMA-7B", 32, 4096, 32, 2048),
    ("LLaMA-13B", 40, 5120, 40, 2048),
    ("LLaMA-70B", 80, 8192, 64, 2048),
]

print("Activation Memory vs Batch Size")
print("=" * 80)
print(f"{'Model':<18} {'B=1':>10} {'B=2':>10} {'B=4':>10} {'B=8':>10} {'B=16':>10}")
print("-" * 80)

for name, L, H, A, S in configs:
    mems = []
    for B in [1, 2, 4, 8, 16]:
        m = activation_memory_gb(L, B, S, H, A)
        mems.append(f"{m:.1f} GB")
    print(f"{name:<18} {mems[0]:>10} {mems[1]:>10} {mems[2]:>10} {mems[3]:>10} {mems[4]:>10}")

print("\nKey insight: Activation memory scales LINEARLY with batch size.")
print("Double the batch size = double the activation memory.")

In [ ]:
#@title 🎧 Transition: Empirical Measurement Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_empirical_measurement_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Let's Build It -- Component by Component

### Empirically Measuring Activation Memory

Let us build a model and actually measure how much memory the activations consume.

In [ ]:
#@title 🎧 Code Walkthrough: Transformer Model
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_transformer_model.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Build a transformer model for measurement

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim=512, num_heads=8, ffn_mult=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * ffn_mult),
            nn.GELU(),
            nn.Linear(hidden_dim * ffn_mult, hidden_dim),
        )

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x

class MeasurableTransformer(nn.Module):
    def __init__(self, hidden_dim=512, num_layers=6, num_heads=8):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

print("Model architecture defined. Ready for memory measurement.")

In [ ]:
#@title 🎧 Code Walkthrough: Measure Activation Memory Func
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_measure_activation_memory_func.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Measure activation memory by comparing memory before/after forward pass

def measure_activation_memory(hidden_dim=512, num_layers=6, num_heads=8,
                                batch_size=4, seq_len=256):
    """Measure activation memory by tracking GPU memory during forward pass."""
    if not torch.cuda.is_available():
        print("GPU required for this measurement.")
        return None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = MeasurableTransformer(hidden_dim, num_layers, num_heads).to(device)

    # Memory after loading model (just weights)
    mem_model = torch.cuda.memory_allocated()

    # Create input
    x = torch.randn(batch_size, seq_len, hidden_dim, device=device)
    mem_input = torch.cuda.memory_allocated()

    # Forward pass with gradient tracking (activations are saved)
    output = model(x)
    loss = output.sum()
    mem_forward = torch.cuda.memory_allocated()

    # Activation memory = total after forward - model - input
    activation_bytes = mem_forward - mem_input

    # Also measure peak during backward
    loss.backward()
    mem_peak = torch.cuda.max_memory_allocated()

    result = {
        'model_mb': (mem_model) / 1e6,
        'input_mb': (mem_input - mem_model) / 1e6,
        'activation_mb': activation_bytes / 1e6,
        'peak_mb': mem_peak / 1e6,
    }

    del model, x, output, loss
    torch.cuda.empty_cache()

    return result

# Measure for a specific configuration
result = measure_activation_memory(
    hidden_dim=512, num_layers=6, num_heads=8,
    batch_size=4, seq_len=256
)

if result:
    print("Memory Breakdown:")
    print(f"  Model weights:   {result['model_mb']:.1f} MB")
    print(f"  Input tensor:    {result['input_mb']:.1f} MB")
    print(f"  Activations:     {result['activation_mb']:.1f} MB")
    print(f"  Peak (fwd+bwd):  {result['peak_mb']:.1f} MB")

In [ ]:
#@title 🎧 What to Look For: Sweep Batch Sizes
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_sweep_batch_sizes.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Sweep over batch sizes to verify linear scaling

if torch.cuda.is_available():
    batch_sizes = [1, 2, 4, 8, 16, 32]
    activation_mems = []

    print(f"{'Batch Size':<15} {'Activation Memory (MB)':<25} {'Ratio to B=1'}")
    print("-" * 55)

    base_mem = None
    for bs in batch_sizes:
        try:
            result = measure_activation_memory(
                hidden_dim=512, num_layers=6, num_heads=8,
                batch_size=bs, seq_len=256
            )
            if result:
                mem = result['activation_mb']
                if base_mem is None:
                    base_mem = mem
                activation_mems.append(mem)
                ratio = mem / base_mem if base_mem > 0 else 0
                print(f"{bs:<15} {mem:<25.1f} {ratio:.2f}x")
        except RuntimeError as e:
            if 'out of memory' in str(e):
                print(f"{bs:<15} OOM!")
                torch.cuda.empty_cache()
                break
            else:
                raise

    # Plot
    if len(activation_mems) > 1:
        fig, ax = plt.subplots(figsize=(10, 5))
        used_bs = batch_sizes[:len(activation_mems)]
        ax.plot(used_bs, activation_mems, 'bo-', markersize=8, linewidth=2, label='Measured')

        # Plot theoretical linear line
        slope = activation_mems[0]
        theoretical = [slope * bs for bs in used_bs]
        ax.plot(used_bs, theoretical, 'r--', linewidth=1.5, alpha=0.7, label='Theoretical (linear)')

        ax.set_xlabel('Batch Size', fontsize=12)
        ax.set_ylabel('Activation Memory (MB)', fontsize=12)
        ax.set_title('Activation Memory vs Batch Size', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        print("\nResult: Activation memory scales linearly with batch size.")
else:
    print("GPU not available. Showing theoretical calculations instead.")
    batch_sizes = [1, 2, 4, 8, 16, 32]
    for bs in batch_sizes:
        mem = activation_memory_gb(6, bs, 256, 512, 8) * 1024  # GB -> MB
        print(f"Batch size {bs:>3}: {mem:.1f} MB (theoretical)")

In [ ]:
#@title 🎧 What to Look For: Sweep Sequence Lengths
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_sweep_sequence_lengths.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Sweep over sequence lengths to verify linear scaling

if torch.cuda.is_available():
    seq_lens = [64, 128, 256, 512, 1024]
    seq_mems = []

    print(f"{'Seq Length':<15} {'Activation Memory (MB)':<25} {'Ratio to S=64'}")
    print("-" * 55)

    base_mem = None
    for sl in seq_lens:
        try:
            result = measure_activation_memory(
                hidden_dim=512, num_layers=6, num_heads=8,
                batch_size=4, seq_len=sl
            )
            if result:
                mem = result['activation_mb']
                if base_mem is None:
                    base_mem = mem
                seq_mems.append(mem)
                ratio = mem / base_mem if base_mem > 0 else 0
                print(f"{sl:<15} {mem:<25.1f} {ratio:.2f}x")
        except RuntimeError as e:
            if 'out of memory' in str(e):
                print(f"{sl:<15} OOM!")
                torch.cuda.empty_cache()
                break
            else:
                raise

    if len(seq_mems) > 1:
        fig, ax = plt.subplots(figsize=(10, 5))
        used_sl = seq_lens[:len(seq_mems)]
        ax.plot(used_sl, seq_mems, 'go-', markersize=8, linewidth=2, label='Measured')
        ax.set_xlabel('Sequence Length', fontsize=12)
        ax.set_ylabel('Activation Memory (MB)', fontsize=12)
        ax.set_title('Activation Memory vs Sequence Length', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        print("\nNote: Scaling is approximately linear but includes an S^2 term from attention scores.")
else:
    print("GPU not available. See theoretical calculations above.")

In [ ]:
#@title 🎧 Listen: Mixed Precision Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_mixed_precision_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Mixed Precision Training

Now let us implement mixed precision training and measure how it affects memory.

The idea: run forward and backward passes in fp16 (fast, less memory), but keep the optimizer in fp32 (numerically stable).

In [ ]:
#@title 🎧 Code Walkthrough: Simple Model
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_simple_model.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate mixed precision training with torch.cuda.amp

from torch.cuda.amp import autocast, GradScaler

class SimpleModel(nn.Module):
    """A simple model for training demonstrations."""
    def __init__(self, vocab_size=10000, hidden_dim=512, num_layers=4, num_heads=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads)
            for _ in range(num_layers)
        ])
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(x)

print("SimpleModel defined for mixed precision experiments.")

In [ ]:
#@title 🎧 Code Walkthrough: Training Functions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_training_functions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Compare fp32 training vs mixed precision training

def train_step_fp32(model, optimizer, input_ids, targets):
    """Standard fp32 training step."""
    optimizer.zero_grad()
    output = model(input_ids)
    loss = nn.functional.cross_entropy(
        output.view(-1, output.size(-1)),
        targets.view(-1)
    )
    loss.backward()
    optimizer.step()
    return loss.item()

def train_step_mixed(model, optimizer, scaler, input_ids, targets):
    """Mixed precision training step."""
    optimizer.zero_grad()

    # Forward pass in fp16
    with autocast(dtype=torch.float16):
        output = model(input_ids)
        loss = nn.functional.cross_entropy(
            output.view(-1, output.size(-1)),
            targets.view(-1)
        )

    # Backward pass: scaler handles fp16 gradient scaling
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    return loss.item()

print("Training functions defined.")
print("")
print("Key difference:")
print("  fp32:  model(x) -> loss -> backward -> step")
print("  mixed: autocast(model(x)) -> scaler.scale(loss).backward -> scaler.step")

In [ ]:
#@title 🎧 Narration: Measure Memory Mixed Precision
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_measure_memory_mixed_precision.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Measure memory for both approaches

if torch.cuda.is_available():
    batch_size = 8
    seq_len = 256
    vocab_size = 10000

    # Create dummy data
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
    targets = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)

    # --- FP32 Training ---
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model_fp32 = SimpleModel(vocab_size=vocab_size).to(device)
    opt_fp32 = torch.optim.Adam(model_fp32.parameters(), lr=1e-4)

    loss_fp32 = train_step_fp32(model_fp32, opt_fp32, input_ids, targets)
    peak_fp32 = torch.cuda.max_memory_allocated() / 1e6

    del model_fp32, opt_fp32
    torch.cuda.empty_cache()

    # --- Mixed Precision Training ---
    torch.cuda.reset_peak_memory_stats()

    model_mixed = SimpleModel(vocab_size=vocab_size).to(device)
    opt_mixed = torch.optim.Adam(model_mixed.parameters(), lr=1e-4)
    scaler = GradScaler()

    loss_mixed = train_step_mixed(model_mixed, opt_mixed, scaler, input_ids, targets)
    peak_mixed = torch.cuda.max_memory_allocated() / 1e6

    del model_mixed, opt_mixed, scaler
    torch.cuda.empty_cache()

    # Report
    print("Memory Comparison: fp32 vs Mixed Precision")
    print("=" * 50)
    print(f"  Config: batch_size={batch_size}, seq_len={seq_len}")
    print(f"")
    print(f"  fp32 peak memory:  {peak_fp32:.1f} MB")
    print(f"  mixed peak memory: {peak_mixed:.1f} MB")
    print(f"  Savings: {(1 - peak_mixed/peak_fp32)*100:.1f}%")
    print(f"")
    print(f"  fp32 loss:  {loss_fp32:.4f}")
    print(f"  mixed loss: {loss_mixed:.4f}")
else:
    print("GPU required for memory comparison. Showing theoretical analysis:")
    print("  Mixed precision saves ~30-50% peak memory by using fp16 activations.")

In [ ]:
#@title 🎧 What to Look For: Full Training Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_full_training_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Full training comparison: fp32 vs mixed precision

import time

if torch.cuda.is_available():
    NUM_STEPS = 50
    batch_size = 8
    seq_len = 128
    vocab_size = 10000

    # --- Train fp32 ---
    torch.cuda.empty_cache()
    model_fp32 = SimpleModel(vocab_size=vocab_size).to(device)
    opt_fp32 = torch.optim.Adam(model_fp32.parameters(), lr=1e-4)

    losses_fp32 = []
    start = time.time()
    for step in range(NUM_STEPS):
        x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        t = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        l = train_step_fp32(model_fp32, opt_fp32, x, t)
        losses_fp32.append(l)
    time_fp32 = time.time() - start

    del model_fp32, opt_fp32
    torch.cuda.empty_cache()

    # --- Train mixed ---
    model_mixed = SimpleModel(vocab_size=vocab_size).to(device)
    opt_mixed = torch.optim.Adam(model_mixed.parameters(), lr=1e-4)
    scaler = GradScaler()

    losses_mixed = []
    start = time.time()
    for step in range(NUM_STEPS):
        x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        t = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        l = train_step_mixed(model_mixed, opt_mixed, scaler, x, t)
        losses_mixed.append(l)
    time_mixed = time.time() - start

    del model_mixed, opt_mixed, scaler
    torch.cuda.empty_cache()

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(losses_fp32, label='fp32', color='#2196F3', linewidth=2)
    axes[0].plot(losses_mixed, label='Mixed Precision', color='#FF5722', linewidth=2)
    axes[0].set_xlabel('Step', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training Loss Comparison', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    bars = axes[1].bar(['fp32', 'Mixed'], [time_fp32, time_mixed],
                        color=['#2196F3', '#FF5722'])
    axes[1].set_ylabel('Time (seconds)', fontsize=12)
    axes[1].set_title(f'Training Time ({NUM_STEPS} steps)', fontsize=13, fontweight='bold')
    for bar, t in zip(bars, [time_fp32, time_mixed]):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{t:.2f}s', ha='center', fontsize=11)
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    speedup = time_fp32 / time_mixed
    print(f"\nSpeedup from mixed precision: {speedup:.2f}x")
    print(f"Both approaches converge similarly -- mixed precision maintains quality.")
else:
    print("GPU required for training comparison.")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn -- TODO Exercises

### TODO 1: Measure Activation Memory vs Number of Layers

We measured how activations scale with batch size and sequence length. Now verify that they also scale linearly with the number of layers. Use `measure_activation_memory()` with `num_layers` ranging from 2 to 12.

In [ ]:
#@title 🎧 Before You Start: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo_1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Sweep over number of layers and measure activation memory

if torch.cuda.is_available():
    layer_counts = [2, 4, 6, 8, 10, 12]
    layer_mems = []

    for n_layers in layer_counts:
        # TODO: Call measure_activation_memory with different num_layers
        # result = measure_activation_memory(
        #     hidden_dim=512, num_layers=???, num_heads=8,
        #     batch_size=4, seq_len=256
        # )
        # layer_mems.append(result['activation_mb'])
        pass  # REPLACE THIS

    # TODO: Create a plot showing activation memory vs number of layers
    # plt.figure(figsize=(10, 5))
    # ... YOUR CODE HERE ...
    # plt.show()

    # TODO: Print whether the scaling is linear
else:
    print("GPU required. Theory predicts linear scaling with layers.")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Loss Scaling Exploration

The `GradScaler` multiplies the loss by a large factor to prevent gradient underflow in fp16. Explore what happens with and without loss scaling.

Train for 30 steps with mixed precision but **without** the GradScaler. Compare the loss curve to training **with** the scaler.

In [ ]:
#@title 🎧 Before You Start: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo_2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Compare mixed precision with and without GradScaler

if torch.cuda.is_available():
    NUM_STEPS = 30
    vocab_size = 10000
    batch_size = 8
    seq_len = 128

    # --- WITH GradScaler (standard approach) ---
    model_scaled = SimpleModel(vocab_size=vocab_size).to(device)
    opt_scaled = torch.optim.Adam(model_scaled.parameters(), lr=1e-4)
    scaler = GradScaler()
    losses_scaled = []

    for step in range(NUM_STEPS):
        x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        t = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        l = train_step_mixed(model_scaled, opt_scaled, scaler, x, t)
        losses_scaled.append(l)

    del model_scaled, opt_scaled, scaler
    torch.cuda.empty_cache()

    # --- WITHOUT GradScaler ---
    model_noscale = SimpleModel(vocab_size=vocab_size).to(device)
    opt_noscale = torch.optim.Adam(model_noscale.parameters(), lr=1e-4)
    losses_noscale = []

    # TODO: Implement training WITHOUT GradScaler
    # Use autocast but call loss.backward() directly (no scaler)
    # and optimizer.step() directly (no scaler)
    for step in range(NUM_STEPS):
        x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
        t = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)

        opt_noscale.zero_grad()
        # TODO: Forward pass with autocast, backward, step -- but NO scaler
        pass  # REPLACE THIS

    del model_noscale, opt_noscale
    torch.cuda.empty_cache()

    # TODO: Plot both loss curves on the same plot
    # Compare: does training without loss scaling still work?
else:
    print("GPU required for this exercise.")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Complete Memory Budget

Combine everything from Notebook 1 and this notebook to create a **complete** memory budget function that includes all four components: weights, gradients, optimizer states, and activations.

In [ ]:
#@title 🎧 Before You Start: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_todo_3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Build the complete memory budget calculator

def complete_memory_budget_gb(
    num_params,
    num_layers,
    hidden_dim,
    num_heads,
    batch_size,
    seq_len,
    precision="mixed",
):
    """Calculate complete training memory budget in GB.

    Returns a dict with memory for each component.
    """
    # TODO: Calculate weight memory
    # Hint: 2 bytes/param for fp16/mixed, 4 for fp32
    weights_gb = 0  # REPLACE

    # TODO: Calculate gradient memory (same size as weights)
    gradients_gb = 0  # REPLACE

    # TODO: Calculate optimizer memory
    # Hint: mixed = 12 bytes/param (master + m + v)
    #        fp32 = 8 bytes/param (m + v)
    optimizer_gb = 0  # REPLACE

    # TODO: Calculate activation memory using the formula
    # Hint: L * B * S * H * 34 bytes
    activations_gb = 0  # REPLACE

    total = weights_gb + gradients_gb + optimizer_gb + activations_gb

    return {
        "weights": weights_gb,
        "gradients": gradients_gb,
        "optimizer": optimizer_gb,
        "activations": activations_gb,
        "total": total,
    }

# Test: GPT-2 (124M), B=8, S=1024
budget = complete_memory_budget_gb(
    num_params=124e6, num_layers=12, hidden_dim=768,
    num_heads=12, batch_size=8, seq_len=1024, precision="mixed"
)

print("GPT-2 (124M), B=8, S=1024, mixed precision:")
for k, v in budget.items():
    print(f"  {k:<15}: {v:.2f} GB")

# Expected total ~4.3 GB (should fit on a T4)
print(f"\nFits on T4 (16GB)? {'Yes' if budget['total'] < 16 else 'No'}")

In [ ]:
#@title 🎧 Transition: Putting It All Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_putting_it_all_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Putting It All Together

Let us create a comprehensive visualization of all four memory components.

In [ ]:
#@title 🎧 What to Look For: Comprehensive Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_comprehensive_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Comprehensive visualization: all four components across batch sizes

def bytes_to_gb(b):
    return b / (1024**3)

# LLaMA-7B config
P = 7e9
L, H, A, S = 32, 4096, 32, 2048

batch_sizes = [1, 2, 4, 8]

weights = [bytes_to_gb(P * 2)] * len(batch_sizes)  # fp16, constant
grads = [bytes_to_gb(P * 2)] * len(batch_sizes)    # fp16, constant
opt = [bytes_to_gb(P * 12)] * len(batch_sizes)     # fp32 master + m + v, constant
acts = [activation_memory_gb(L, B, S, H, A) for B in batch_sizes]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Stacked bar ---
x = np.arange(len(batch_sizes))
width = 0.6

axes[0].bar(x, weights, width, label='Weights (fp16)', color='#2196F3')
axes[0].bar(x, grads, width, bottom=weights, label='Gradients (fp16)', color='#4CAF50')
axes[0].bar(x, opt, width, bottom=[w+g for w,g in zip(weights, grads)],
            label='Optimizer (fp32)', color='#FF5722')
axes[0].bar(x, acts, width,
            bottom=[w+g+o for w,g,o in zip(weights, grads, opt)],
            label='Activations', color='#9C27B0')

# GPU lines
axes[0].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='A100 80GB')
axes[0].axhline(y=40, color='orange', linestyle='--', alpha=0.5, label='A100 40GB')

axes[0].set_xticks(x)
axes[0].set_xticklabels([f'B={b}' for b in batch_sizes])
axes[0].set_ylabel('Memory (GB)', fontsize=12)
axes[0].set_title('LLaMA-7B: Full Training Memory', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# --- Panel 2: Component share at B=1 vs B=8 ---
for idx, B in enumerate([1, 8]):
    b_idx = batch_sizes.index(B)
    components = [weights[b_idx], grads[b_idx], opt[b_idx], acts[b_idx]]
    labels = ['Weights', 'Gradients', 'Optimizer', 'Activations']
    colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

    # Create a horizontal bar
    total = sum(components)
    left = 0
    y_pos = 1 - idx * 0.5
    for comp, label, color in zip(components, labels, colors):
        pct = comp / total * 100
        axes[1].barh(y_pos, comp, left=left, height=0.35, color=color)
        if pct > 8:
            axes[1].text(left + comp/2, y_pos, f'{label}\n{pct:.0f}%',
                        ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        left += comp

    axes[1].text(-5, y_pos, f'B={B}\n{total:.0f}GB', ha='right', va='center', fontsize=11, fontweight='bold')

axes[1].set_xlim(-40, max(sum(c) for c in zip(weights, grads, opt, acts)) * 1.05)
axes[1].set_yticks([])
axes[1].set_xlabel('Memory (GB)', fontsize=12)
axes[1].set_title('Memory Share: Small vs Large Batch', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key insight:")
print(f"  At B=1: Optimizer states dominate ({opt[0]/(weights[0]+grads[0]+opt[0]+acts[0])*100:.0f}% of total)")
print(f"  At B=8: Activations dominate ({acts[3]/(weights[3]+grads[3]+opt[3]+acts[3])*100:.0f}% of total)")
print(f"  This determines which optimization strategy to use!")

In [ ]:
#@title 🎧 Listen: Summary Table
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_summary_table.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Training and Results

Let us summarize the activation memory patterns we observed.

In [ ]:
# Summary table
print("Activation Memory Scaling Summary")
print("=" * 60)
print(f"{'Dimension':<25} {'Scaling':<20} {'Implication'}")
print("-" * 60)
print(f"{'Batch size (B)':<25} {'Linear':<20} Reduce B to save memory")
print(f"{'Sequence length (S)':<25} {'~Linear + S^2':<20} Attention is S^2")
print(f"{'Hidden dimension (H)':<25} {'Linear':<20} Set by architecture")
print(f"{'Number of layers (L)':<25} {'Linear':<20} Set by architecture")
print()
print("Practical Takeaways:")
print("  1. Batch size is the easiest knob to turn when OOM")
print("  2. Mixed precision saves ~30-50% peak memory")
print("  3. For long sequences, attention score memory (S^2) dominates")
print("  4. Activation recomputation trades compute for memory")

In [ ]:
#@title 🎧 Listen: Final Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_final_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Final Output

In [ ]:
print("=" * 60)
print("SUMMARY: Activation Memory and Mixed Precision")
print("=" * 60)
print()
print("Activation Memory Formula:")
print("  M_act = L * B * S * H * 34 bytes (simplified)")
print("  M_act = L * B * S * H * (34 + 5*A*S/H) bytes (full)")
print()
print("Mixed Precision Training:")
print("  1. Cast fp32 master weights to fp16")
print("  2. Forward + backward in fp16 (fast, less memory)")
print("  3. Optimizer update in fp32 (numerically stable)")
print("  4. GradScaler prevents gradient underflow")
print()
print("Complete Training Memory (mixed precision + Adam):")
print("  M_total = 2P + 2P + 12P + M_activations = 16P + M_act")
print()
print("Decision Framework:")
print("  Small batch -> Optimizer dominates -> Use ZeRO")
print("  Large batch -> Activations dominate -> Use gradient checkpointing")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

**Questions to think about:**
1. For a model with S=8192 (long context), how much of the activation memory comes from attention scores vs FFN intermediates?
2. If you could recompute activations during the backward pass instead of storing them, how much memory would you save? What is the cost?
3. Why is bf16 preferred over fp16 for mixed precision training on hardware that supports it?

**What comes next:**
In the next notebook, we will build a **"Will It Fit?" calculator** that takes any model configuration and training setup, computes the complete memory budget, and tells you whether it fits on your GPU -- with recommendations for what to change if it does not.